In [0]:
from pyspark.sql import DataFrame
from functools import reduce
from pyspark.sql.functions import col, lit
from pyspark.sql.types import StructType
from delta.tables import DeltaTable


# ============================================================
# DE-DUPLICATION
# ============================================================


def remove_duplicates(df: DataFrame) -> DataFrame:
    """
    Removes exact duplicate rows from the DataFrame.

    Bronze should preserve raw data.
    Silver removes accidental duplicate records before further processing.
    """
    before_count = df.count()

    dedup_df = df.dropDuplicates()

    after_count = dedup_df.count()

    print(f"Rows before deduplication : {before_count}")
    print(f"Rows after deduplication  : {after_count}")
    print(f"Duplicates removed        : {before_count - after_count}")

    return dedup_df


# ============================================================
# QUALITY CHECK
# ============================================================

def apply_data_quality(
    df: DataFrame,
    required_columns: list = None,
    non_negative_columns: list = None
):
    """
    Splits the DataFrame into valid and invalid records based on
    configurable data quality rules.

    Returns:
        valid_df, invalid_df
    """

    required_columns = required_columns or []
    non_negative_columns = non_negative_columns or []

    conditions = []

    # Required (NOT NULL) columns
    for column in required_columns:
        conditions.append(col(column).isNotNull())

    # Non-negative numeric columns
    for column in non_negative_columns:
        conditions.append(col(column) >= 0)

    # If no rules exist, everything is valid
    if conditions:
        valid_condition = reduce(lambda x, y: x & y, conditions)
    else:
        valid_condition = lit(True)

    valid_df = df.filter(valid_condition)
    invalid_df = df.filter(~valid_condition)

    print(f"Total rows   : {df.count()}")
    print(f"Valid rows   : {valid_df.count()}")
    print(f"Invalid rows : {invalid_df.count()}")

    return valid_df, invalid_df    



# ============================================================
# SCHEMA DRIFT AND EVOLUTION
# ============================================================




def detect_schema_drift(source_df, target_df,system_columns=None):
    """
    Compares source and target schemas.

    Returns:
        {
            "new_columns": [...],
            "missing_columns": [...],
            "type_changes": [...]
        }
    """

    ignore_columns = system_columns or []    


    source_schema = {f.name: f.dataType.simpleString() for f in source_df.schema.fields }
    target_schema = {f.name: f.dataType.simpleString() for f in target_df.schema.fields if f.name not in ignore_columns}

    source_cols = set(source_schema.keys())
    target_cols = set(target_schema.keys())

    new_columns = sorted(list(source_cols - target_cols))
    missing_columns = sorted(list(target_cols - source_cols))

    type_changes = []

    for col_name in source_cols.intersection(target_cols):
        if source_schema[col_name] != target_schema[col_name]:
            type_changes.append({
                "column": col_name,
                "source_type": source_schema[col_name],
                "target_type": target_schema[col_name]
            })

    return {
        "new_columns": new_columns,
        "missing_columns": missing_columns,
        "type_changes": type_changes
    }


def log_schema_drift(table_name, drift):

    print("=" * 60)
    print(f"Schema Drift Report : {table_name}")
    print("=" * 60)

    print(f"New Columns     : {drift['new_columns']}")
    print(f"Missing Columns : {drift['missing_columns']}")

    print("Type Changes")

    if drift["type_changes"]:
        for change in drift["type_changes"]:
            print(
                f"{change['column']} : "
                f"{change['target_type']} -> {change['source_type']}"
            )
    else:
        print("None")

    print("=" * 60)

def validate_schema_drift(drift):
    """
    Returns True if pipeline can continue.
    Raises an exception for unsupported schema drift.
    """

    if drift["missing_columns"]:
        raise Exception(
            f"Missing columns detected: {drift['missing_columns']}"
        )

    if drift["type_changes"]:
        raise Exception(
            f"Data type changes detected: {drift['type_changes']}"
        )

    return True





def create_silver_table_if_not_exists(
    bronze_df,
    silver_table_name,
    silver_path,
    surrogate_key_column
):
    """
    Creates an external Silver Delta table if it does not already exist.
    Business columns are inferred from the Bronze DataFrame schema.
    """

    # Already exists?
    if spark.catalog.tableExists(silver_table_name):
        print(f"✓ {silver_table_name} already exists.")
        return

    bronze_columns = ",\n        ".join(
        [
            f"`{field.name}` {field.dataType.simpleString()}"
            for field in bronze_df.schema.fields
        ]
    )

    create_sql = f"""
    CREATE TABLE {silver_table_name}
    (
        {surrogate_key_column} BIGINT GENERATED ALWAYS AS IDENTITY,

        {bronze_columns},

        EffectiveStartDate TIMESTAMP,
        EffectiveEndDate TIMESTAMP,
        IsCurrent BOOLEAN
    )
    USING DELTA
    LOCATION '{silver_path}'
    """

    spark.sql(create_sql)

    print(f"✓ Created Silver table {silver_table_name}")    


def handle_schema_evolution(
    silver_table_name,
    drift,
    source_df
):
    """
    Adds newly detected business columns to the Silver table.

    Existing rows will contain NULL for the newly added columns.
    """

    new_columns = drift["new_columns"]

    if not new_columns:
        print("✓ No schema evolution required.")
        return

    # Bronze schema lookup
    source_schema = {
        field.name: field.dataType.simpleString()
        for field in source_df.schema.fields
    }

    alter_columns = []

    for column in new_columns:
        datatype = source_schema[column]
        alter_columns.append(f"`{column}` {datatype}")

    alter_sql = f"""
    ALTER TABLE {silver_table_name}
    ADD COLUMNS (
        {", ".join(alter_columns)}
    )
    """

    print("Executing Schema Evolution...")
    print(alter_sql)

    spark.sql(alter_sql)

    print(f"✓ Added {len(new_columns)} column(s) to Silver.")   


# ============================================================
# DETECT SCD CHANGES
# ============================================================


def detect_scd_changes(
    bronze_df,
    silver_df,
    business_key,
    compare_columns
):
    """
    Detect SCD Type 2 changes.

    Returns:
        new_records,
        changed_records,
        unchanged_records
    """

    current_silver = silver_df.filter(col("IsCurrent") == True)

    joined = bronze_df.alias("b").join(
        silver_df.alias("s"),
        on=business_key,
        how="left"
    )

    # New business key
    new_records = joined.filter(
        col(f"s.{business_key}").isNull()
    ).select("b.*")

    # Compare business attributes
    compare_condition = reduce(
        lambda x, y: x | y,
        [
            col(f"b.{c}") != col(f"s.{c}")
            for c in compare_columns
        ]
    )

    changed_records = joined.filter(
        col(f"s.{business_key}").isNotNull() &
        compare_condition
    ).select("b.*")

    unchanged_records = joined.filter(
        col(f"s.{business_key}").isNotNull() &
        (~compare_condition)
    ).select("b.*")

    
    return (
        new_records,
        changed_records,
        unchanged_records
    )


from pyspark.sql.functions import (
    col,
    lit,
    current_timestamp
)

def prepare_merge_source(
    new_records_df,
    changed_records_df,
    business_key,
    business_key_type
):
    """
    Prepare staging DataFrame for SCD Type 2 Delta MERGE.

    UPDATE rows
        - Expire current Silver record
        - MergeKey = BusinessKey

    INSERT rows
        - Insert new current version
        - MergeKey = NULL

    Returns
    -------
    merge_source_df
    """

    HIGH_DATE = "9999-12-31 23:59:59"

    # ==========================================================
    # UPDATE rows (expire current record)
    # ==========================================================

    update_rows = (
        changed_records_df
            .withColumn("MergeKey", col(business_key))
            .withColumn("MergeAction", lit("UPDATE"))
    )

    # ==========================================================
    # INSERT rows (changed records)
    # ==========================================================

    changed_insert_rows = (
        changed_records_df
            .withColumn("MergeKey", lit(None).cast(business_key_type))
            .withColumn("MergeAction", lit("INSERT"))
            .withColumn("EffectiveStartDate", col("LastModifiedDate").cast("timestamp"))            
            .withColumn(
                "EffectiveEndDate",
                lit(HIGH_DATE).cast("timestamp")
            )
            .withColumn("IsCurrent", lit(True))
    )

    # ==========================================================
    # INSERT rows (new records)
    # ==========================================================

    new_insert_rows = (
        new_records_df
            .withColumn("MergeKey", lit(None).cast(business_key_type))
            .withColumn("MergeAction", lit("INSERT"))
            .withColumn("EffectiveStartDate", col("LastModifiedDate").cast("timestamp"))  
            .withColumn(
                "EffectiveEndDate",
                lit(HIGH_DATE).cast("timestamp")
            )
            .withColumn("IsCurrent", lit(True))
    )

    # ==========================================================
    # Final merge source
    # ==========================================================

    merge_source_df = (
        update_rows
            .unionByName(
                changed_insert_rows,
                allowMissingColumns=True
            )
            .unionByName(
                new_insert_rows,
                allowMissingColumns=True
            )
    )

    return merge_source_df



from delta.tables import DeltaTable
from pyspark.sql.functions import current_timestamp, lit


def merge_to_silver(
    merge_source_df,
    silver_table_name,
    business_key
):
    """
    Applies SCD Type 2 using a single atomic Delta MERGE.

    UPDATE rows
        - Expire existing current record

    INSERT rows
        - Insert new business keys
        - Insert changed business keys
    """

    target = DeltaTable.forName(spark, silver_table_name)

    (
        target.alias("target")
        .merge(
            merge_source_df.alias("source"),
            f"""
            target.{business_key} = source.MergeKey
            AND target.IsCurrent = true
            """
        )

        # =====================================================
        # Expire existing current record
        # =====================================================
        .whenMatchedUpdate(
            condition="source.MergeAction = 'UPDATE'",
            set={
                "EffectiveEndDate": "source.LastModifiedDate",
                "IsCurrent": "false"
            }
        )

        # =====================================================
        # Insert new record / changed record
        # =====================================================
        .whenNotMatchedInsert(
            values={
                column: f"source.{column}"
                for column in merge_source_df.columns
                if column not in ["MergeKey", "MergeAction"]
            }
        )

        .execute()
    )

    print("✓ Silver MERGE completed.")



# create_silver_table_if_not_exists(
#     valid_df,
#     silver_table_name,
#     silver_path,
#     surrogate_key_column
# )



def append_to_silver(valid_df,silver_table_name):
    valid_df.write\
        .mode('append')\
        .format('delta')\
        .saveAsTable(silver_table_name)
    print("✓ Silver APPEND completed.")
    


